## 1. Imports e montagem dos headers

Aqui eu só transcrevo para o Python os campos de request do header capturado.

In [15]:
import json
import os
from pathlib import Path

import pandas as pd
import requests

BASE = "https://apis.estudante.ibmec.br/rest"

# token do header "authorization" (sem o "Bearer "), lido de fora do notebook
TOKEN = os.environ.get("IBMEC_TOKEN", "")
if not TOKEN and Path("token.txt").exists():
    TOKEN = Path("token.txt").read_text(encoding="utf-8").strip()

# headers copiados 1:1 da requisicao do navegador
headers = {
    "accept": "application/json, text/plain, */*",
    "accept-language": "pt-BR,pt;q=0.9,en-US;q=0.8,en;q=0.7",
    "authorization": f"Bearer {TOKEN}",
    "origin": "https://estudante.ibmec.br",
    "referer": "https://estudante.ibmec.br/",
    "user-agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 "
        "(KHTML, like Gecko) Chrome/152.0.0.0 Safari/537.36"
    ),
}

print("token carregado" if TOKEN else "sem token - vai usar o cache")

token carregado


## 2. O que tem dentro do token

Um JWT é só `header.payload.assinatura`, cada parte em base64url. Dá para abrir o payload sem nenhuma
biblioteca — **é só decodificar, não é quebrar nada**: o conteúdo é público para quem tem o token, a
assinatura é que impede forjar um novo.

Faço isso por dois motivos práticos: pegar a **matrícula** (`upn`), que alguns endpoints exigem como
parâmetro, e ver o `exp` para saber se o token ainda vale.

In [16]:
import base64
from datetime import datetime, timezone


def le_jwt(token):
    """Decodifica o payload do JWT (base64url, sem validar assinatura)."""
    payload = token.split(".")[1]
    payload += "=" * (-len(payload) % 4)  # base64url vem sem padding
    return json.loads(base64.urlsafe_b64decode(payload))


if TOKEN:
    claims = le_jwt(TOKEN)
    expira = datetime.fromtimestamp(claims["exp"], tz=timezone.utc)
    MATRICULA = claims["upn"].split("@")[0]
    print("aluno    :", claims["name"])
    print("matricula:", MATRICULA)
    print("expira em:", expira.astimezone(), "->", "valido" if expira > datetime.now(timezone.utc) else "VENCIDO")
else:
    MATRICULA = None

aluno    : EDUARDO PALHARES REALE PEREIRA
matricula: 202507222597
expira em: 2026-09-09 08:02:24-03:00 -> valido


## 3. Requisição das disciplinas

Faço **GET**, e não OPTIONS: o preflight é coisa do navegador, o `requests` não faz. Se a chamada
falhar (token vencido → `401`/`403`), caio no `turmas-atual.json` da última extração.

In [17]:
CACHE_TURMAS = Path("turmas-atual.json")


def busca_turmas():
    """Tenta a API; se falhar, usa o JSON salvo da ultima extracao."""
    try:
        resposta = requests.get(
            f"{BASE}/turmas/status", params={"status": "ATUAL"}, headers=headers, timeout=20
        )
        resposta.raise_for_status()
        dados = resposta.json()
        CACHE_TURMAS.write_text(json.dumps(dados, ensure_ascii=False, indent=2), encoding="utf-8")
        print(f"API respondeu {resposta.status_code} - {len(dados)} turmas")
        return dados
    except Exception as erro:
        print(f"Requisicao falhou ({type(erro).__name__}: {erro}) - usando {CACHE_TURMAS}")
        return json.loads(CACHE_TURMAS.read_text(encoding="utf-8"))


turmas = busca_turmas()

API respondeu 200 - 5 turmas


## 4. Como vem cada turma no JSON

Olho a estrutura de um registro para saber quais chaves viram coluna. `local`, `educadores` e
`horarios` são aninhados — é o que precisa de tratamento.

In [18]:
turmas[0]

{'id': 'ibmec_14736964',
 'idExterno': None,
 'marca': 'IBMEC',
 'codigoEntrega': '8001',
 'formato': 'presencial',
 'campus': 'BELO HORIZONTE - FUNCIONÁRIOS',
 'tipoCurso': 'GRADUAÇÃO',
 'educadorResponsavel': {'nome': 'ANGELICA MATOS GUIMARAES DIAS',
  'perfil': 'professor'},
 'periodoAcademico': '2026.2',
 'local': {'blocos': ['1', '1', '1', '1'],
  'salas': ['308', '308', '308', '308']},
 'horarios': [{'diaSemana': 'Qua',
   'horaInicio': '12:50:00Z',
   'horaFim': '13:45:00Z'},
  {'diaSemana': 'Qua', 'horaInicio': '13:45:00Z', 'horaFim': '14:40:00Z'},
  {'diaSemana': 'Qui', 'horaInicio': '12:50:00Z', 'horaFim': '13:45:00Z'},
  {'diaSemana': 'Qui', 'horaInicio': '13:45:00Z', 'horaFim': '14:40:00Z'}],
 'educadores': [{'nome': 'ANGELICA MATOS GUIMARAES DIAS',
   'perfil': 'professor'},
  {'nome': 'ANGELICA MATOS GUIMARAES DIAS', 'perfil': 'professor'},
  {'nome': 'ANGELICA MATOS GUIMARAES DIAS', 'perfil': 'professor'},
  {'nome': 'ANGELICA MATOS GUIMARAES DIAS', 'perfil': 'professor'

## 5. Achatando os campos aninhados

Três coisas precisam de tratamento antes de virar coluna:

- **`horarios`** — o portal quebra cada aula em blocos de 55 min (`12:50–13:45` + `13:45–14:40`).
  A função abaixo junta os blocos seguidos do mesmo dia num intervalo só: `Qua 12:50-14:40`.
- **`local.blocos` / `local.salas`** — vêm repetidos (um por bloco de aula) ⇒ tiro as duplicatas.
- **`educadores`** — idem, uso `set` para não repetir o nome do professor.

In [19]:
def formata_horarios(horarios):
    """Junta as aulas seguidas do mesmo dia num intervalo unico: 'Qua 12:50-14:40'."""
    por_dia = {}
    for h in horarios or []:
        dia = h["diaSemana"]
        ini, fim = h["horaInicio"][:5], h["horaFim"][:5]
        if dia in por_dia:
            por_dia[dia] = (min(por_dia[dia][0], ini), max(por_dia[dia][1], fim))
        else:
            por_dia[dia] = (ini, fim)
    return "; ".join(f"{d} {i}-{f}" for d, (i, f) in por_dia.items())


linhas = []
for t in turmas:
    local = t.get("local") or {}
    professores = sorted({e["nome"] for e in t.get("educadores") or []})
    linhas.append(
        {
            "codigo_disciplina": t["codigoDisciplina"],
            "disciplina": t["nome"],
            "periodo": t["periodoAcademico"],
            "professor_responsavel": (t.get("educadorResponsavel") or {}).get("nome"),
            "professores": ", ".join(professores),
            "horarios": formata_horarios(t.get("horarios")),
            "bloco": ", ".join(sorted(set(local.get("blocos") or []))),
            "sala": ", ".join(sorted(set(local.get("salas") or []))),
            "campus": t["campus"],
            "formato": t["formato"],
            "tipo_curso": t["tipoCurso"],
            "alunos_matriculados": t["totalAlunosMatriculados"],
            "turma_id": t["id"],
        }
    )

df = pd.DataFrame(linhas).sort_values("disciplina").reset_index(drop=True)
print(df.shape)

(5, 13)


## 6. Dataframe das disciplinas

Uma linha por disciplina matriculada no período atual.

In [20]:
df

,codigo_disciplina,disciplina,periodo,professor_responsavel,professores,horarios,bloco,sala,campus,formato,tipo_curso,alunos_matriculados,turma_id
0,IBM8915,EXTRAÇÃO E PREPARAÇÃO DE DADOS,2026.2,PEDRO HENRIQUE CALAIS GUERRA,PEDRO HENRIQUE CALAIS GUERRA,Ter 12:50-14:40; Qua 10:30-12:20,1,108,BELO HORIZONTE - FUNCIONÁRIOS,presencial,GRADUAÇÃO,16,ibmec_14736977
1,IBM0037,INFERÊNCIA ESTATÍSTICA,2026.2,FRANK MAGALHAES DE PINHO,FRANK MAGALHAES DE PINHO,Seg 10:30-12:20; Qui 10:30-12:20,1,313,BELO HORIZONTE - FUNCIONÁRIOS,presencial,GRADUAÇÃO,48,ibmec_14736680
2,IBM1740,INOVAÇÃO E DESIGN THINKING,2026.2,TADEU MOREIRA PERONA,TADEU MOREIRA PERONA,Ter 10:30-12:20; Sex 10:30-12:20,2,206,BELO HORIZONTE - FUNCIONÁRIOS,presencial,GRADUAÇÃO,21,ibmec_14736847
3,IBM0792,MÉTODOS ÁGEIS DE DESENVOLVIMENTO DE SOFTWARE,2026.2,EDES GARCIA DA COSTA FILHO,EDES GARCIA DA COSTA FILHO,Seg 12:50-14:40; Sex 12:50-14:40,1,112,BELO HORIZONTE - FUNCIONÁRIOS,presencial,GRADUAÇÃO,52,ibmec_14736836
4,IBM4028,PROJETO EM CIÊNCIA DE DADOS IV,2026.2,ANGELICA MATOS GUIMARAES DIAS,ANGELICA MATOS GUIMARAES DIAS,Qua 12:50-14:40; Qui 12:50-14:40,1,308,BELO HORIZONTE - FUNCIONÁRIOS,presencial,GRADUAÇÃO,8,ibmec_14736964


## 7. Uma linha por aula (grade da semana)

O dataframe acima tem os horários concatenados em texto — bom para ler, ruim para analisar. Aqui uso
`explode` + `json_normalize` para abrir a lista `horarios` em uma linha por aula, e depois pivoto
para montar a grade.

In [21]:
df_horarios = (
    pd.DataFrame(turmas)[["codigoDisciplina", "nome", "horarios"]]
    .explode("horarios")
    .dropna(subset=["horarios"])
)
df_horarios = pd.concat(
    [
        df_horarios.drop(columns="horarios").reset_index(drop=True),
        pd.json_normalize(df_horarios["horarios"]).reset_index(drop=True),
    ],
    axis=1,
).rename(columns={"codigoDisciplina": "codigo_disciplina", "nome": "disciplina"})

DIAS = ["Seg", "Ter", "Qua", "Qui", "Sex", "Sab"]
grade = df_horarios.copy()
grade["faixa"] = grade["horaInicio"].str[:5] + "-" + grade["horaFim"].str[:5]
grade = (
    grade.pivot_table(index="faixa", columns="diaSemana", values="disciplina", aggfunc="first")
    .reindex(columns=[d for d in DIAS if d in grade["diaSemana"].values])
    .fillna("")
)
grade

diaSemana,Seg,Ter,Qua,Qui,Sex
faixa,,,,,
10:30-11:25,INFERÊNCIA ESTATÍSTICA,INOVAÇÃO E DESIGN THINKING,EXTRAÇÃO E PREPARAÇÃO DE DADOS,INFERÊNCIA ESTATÍSTICA,INOVAÇÃO E DESIGN THINKING
11:25-12:20,INFERÊNCIA ESTATÍSTICA,INOVAÇÃO E DESIGN THINKING,EXTRAÇÃO E PREPARAÇÃO DE DADOS,INFERÊNCIA ESTATÍSTICA,INOVAÇÃO E DESIGN THINKING
12:50-13:45,MÉTODOS ÁGEIS DE DESENVOLVIMENTO DE SOFTWARE,EXTRAÇÃO E PREPARAÇÃO DE DADOS,PROJETO EM CIÊNCIA DE DADOS IV,PROJETO EM CIÊNCIA DE DADOS IV,MÉTODOS ÁGEIS DE DESENVOLVIMENTO DE SOFTWARE
13:45-14:40,MÉTODOS ÁGEIS DE DESENVOLVIMENTO DE SOFTWARE,EXTRAÇÃO E PREPARAÇÃO DE DADOS,PROJETO EM CIÊNCIA DE DADOS IV,PROJETO EM CIÊNCIA DE DADOS IV,MÉTODOS ÁGEIS DE DESENVOLVIMENTO DE SOFTWARE


---

# Parte 2 — Atividades: nome, prazo e se foi entregue

## O problema

O endpoint `/rest/turmas/status?status=ATUAL` **não traz atividade nenhuma**. Ele devolve só turma,
professor, horário, sala e um `possuiExercicios: false`. Não existe ali nome de atividade, prazo nem
status de entrega. Ou seja: o header capturado me deu a autenticação e o padrão das chamadas, mas a
lista de atividades vem de **outro endpoint**, que eu não tinha.

## Como descobri o endpoint certo (sem ficar chutando URL)

Chutar caminho na API é ruim: é lento, é barulhento no servidor e provavelmente não acerta. Fui pela
porta da frente — **o próprio front-end do portal sabe as rotas, e o código dele é público**:

1. Baixei o HTML de `https://estudante.ibmec.br` e peguei os `<script src>`:
   `main.js`, `aura.js`, `lift.js`, `vendors.js`.
2. As chamadas não estavam nos bundles principais — o app usa *code splitting*, então extraí do
   `main.js` o mapa de chunks do webpack (`{49:"35b42f4a", 60:"a3dc14a6", ...}`, 101 chunks) e baixei
   todos de `/static/js/{id}.{hash}.chunk.js`.
3. Procurei nos arquivos o padrão com que o app monta as URLs — `"".concat(urlBase, "/algumaCoisa")`
   junto de `authorization: "Bearer ".concat(token)` — e aí apareceu a lista real de rotas.

As que interessam:

| Rota descoberta | O que faz |
|---|---|
| `GET /trabalhos/turmas/{idTurma}` | devolve a **lista de ids** das atividades daquela turma |
| `GET /trabalhos/{idTrabalho}?idTurma={idTurma}` | devolve o **detalhe**: título, prazo, status da entrega |
| `GET /exercicios/turmas/{idTurma}` | exercícios (nessas turmas volta vazio) |
| `GET /me/disciplinas/avaliacoes/status` | avaliações formais (também vazio no período) |

## De onde sai o check verde e o X vermelho

O detalhe de cada atividade traz `statusResposta`. Achei no bundle o dicionário que o portal usa para
pintar a tag na tela, e é exatamente esse o significado:

```js
{ NAO_RESPONDIDO: "Pendente",  RESPONDIDO: "Enviado/Concluído",
  CORRIGIDO:      "Corrigido", FALHA_DE_ENVIO: "Pendente" }
```

Então a regra do ✅ / ❌ não é invenção minha, é a mesma do portal:

- ✅ **entregue** → `statusResposta` é `RESPONDIDO` ou `CORRIGIDO`
- ❌ **não entregue** → `NAO_RESPONDIDO` ou `FALHA_DE_ENVIO`

## 8. Buscando as atividades

Duas chamadas em cascata: para cada turma, pego a lista de ids; para cada id, pego o detalhe.
São 5 turmas ⇒ 5 + N requisições. Guardo o resultado cru em `trabalhos-atual.json` para servir de
cache quando o token vencer.

In [22]:
CACHE_TRAB = Path("trabalhos-atual.json")


def busca_atividades(turmas):
    """Para cada turma: lista os ids das atividades e busca o detalhe de cada uma."""
    coletadas = []
    for t in turmas:
        ids = requests.get(
            f"{BASE}/trabalhos/turmas/{t['id']}", headers=headers, timeout=25
        ).json()
        print(f"  {t['codigoDisciplina']:8s} {t['nome'][:42]:42s} -> {len(ids):2d} atividades")
        for id_trabalho in ids:
            detalhe = requests.get(
                f"{BASE}/trabalhos/{id_trabalho}",
                headers=headers,
                params={"idTurma": t["id"]},
                timeout=25,
            ).json()
            detalhe["codigo_disciplina"] = t["codigoDisciplina"]
            detalhe["disciplina"] = t["nome"]
            coletadas.append(detalhe)
    return coletadas


try:
    atividades = busca_atividades(turmas)
    CACHE_TRAB.write_text(json.dumps(atividades, ensure_ascii=False, indent=2), encoding="utf-8")
except Exception as erro:
    print(f"Falhou ({type(erro).__name__}: {erro}) - usando {CACHE_TRAB}")
    atividades = json.loads(CACHE_TRAB.read_text(encoding="utf-8"))

print(f"\n{len(atividades)} atividades no total")

  IBM4028  PROJETO EM CIÊNCIA DE DADOS IV             ->  1 atividades
  IBM8915  EXTRAÇÃO E PREPARAÇÃO DE DADOS             -> 12 atividades
  IBM0037  INFERÊNCIA ESTATÍSTICA                     ->  0 atividades
  IBM0792  MÉTODOS ÁGEIS DE DESENVOLVIMENTO DE SOFTWA ->  0 atividades
  IBM1740  INOVAÇÃO E DESIGN THINKING                 ->  2 atividades

15 atividades no total


## 9. Como vem uma atividade

Os três campos que eu quero: `titulo` (nome), `prazoEntrega` (data) e `statusResposta` (entregue ou não).

In [23]:
atividades[0]

{'id': '6a86ee4fd21822eaa3824c08',
 'entregas': ['ibmec_14736964'],
 'titulo': 'ATIVIDADE —QUALIDADE DE SOFTWARE',
 'enunciado': 'Conforme anexo',
 'tipoTrabalho': 'GRUPO',
 'modelo': 'COMUM',
 'obrigatorio': True,
 'avaliacao': {'tipo': 'NENHUMA_CORRECAO'},
 'prazoEntrega': '2026-08-26T23:59:41.000Z',
 'permitirEnvioForaPrazo': False,
 'quantidadeMateriaisApoio': 1,
 'temResposta': True,
 'dataAtualizacao': '2026-08-21T14:04:31.818Z',
 'dataCriacao': '2026-08-20T12:08:47.479Z',
 'statusResposta': 'RESPONDIDO',
 'codigo_disciplina': 'IBM4028',
 'disciplina': 'PROJETO EM CIÊNCIA DE DADOS IV'}

## 10. Montando a tabela

- **`prazoEntrega`** vem em ISO UTC (`2026-09-10T23:59:00.000Z`) ⇒ converto para `datetime` e
  formato como `dd/mm/aaaa`. Algumas atividades podem vir sem prazo, então trato o nulo.
- **`statusResposta`** ⇒ aplico a regra do portal para virar ✅ ou ❌.

In [24]:
ENTREGUE = {"RESPONDIDO", "CORRIGIDO"}          # portal mostra "Enviado" / "Corrigido"
NAO_ENTREGUE = {"NAO_RESPONDIDO", "FALHA_DE_ENVIO"}  # portal mostra "Pendente"

registros = []
for a in atividades:
    prazo = pd.to_datetime(a.get("prazoEntrega"), format="ISO8601", utc=True, errors="coerce")
    status = a.get("statusResposta")
    registros.append(
        {
            "disciplina": a["disciplina"],
            "nome": a["titulo"],
            "data": prazo,
            "entregue": "\u2705" if status in ENTREGUE else "\u274c",
            "status_bruto": status,
            "obrigatorio": a.get("obrigatorio"),
            "tipo": a.get("tipoTrabalho"),
        }
    )

df_atividades = (
    pd.DataFrame(registros).sort_values(["data", "disciplina"]).reset_index(drop=True)
)
df_atividades["data"] = df_atividades["data"].dt.strftime("%d/%m/%Y")

print(df_atividades.shape)

(15, 7)


## 11. Tabela pedida — Nome, Data, Entregue

`✅` = atividade entregue, `❌` = não entregue.

In [25]:
df_atividades[["nome", "data", "entregue"]]

,nome,data,entregue
0,AC 1 - Extracao Tabela Tabula,12/08/2026,✅
1,AC 2 - Camelot,18/08/2026,✅
2,AC 3 - Tabela Guns,19/08/2026,✅
3,AC Pensamento Critico,20/08/2026,✅
4,Roda da inovação,24/08/2026,✅
5,AC 4 - Coleta Tabela McDonalds,25/08/2026,✅
6,Formulário,25/08/2026,✅
7,ATIVIDADE —QUALIDADE DE SOFTWARE,26/08/2026,✅
8,AC 6 - Pontos vs Saldo com Escudo - Serie A Br...,01/09/2026,✅
9,AC 7 - Nuvem de Palavras Frases Motivacionais,02/09/2026,✅


## 12. A mesma tabela com a disciplina e o status original

Mantenho `status_bruto` ao lado do ✅/❌ para deixar auditável de onde saiu cada símbolo.

In [26]:
df_atividades

,disciplina,nome,data,entregue,status_bruto,obrigatorio,tipo
0,EXTRAÇÃO E PREPARAÇÃO DE DADOS,AC 1 - Extracao Tabela Tabula,12/08/2026,✅,RESPONDIDO,True,INDIVIDUAL
1,EXTRAÇÃO E PREPARAÇÃO DE DADOS,AC 2 - Camelot,18/08/2026,✅,RESPONDIDO,True,INDIVIDUAL
2,EXTRAÇÃO E PREPARAÇÃO DE DADOS,AC 3 - Tabela Guns,19/08/2026,✅,RESPONDIDO,True,INDIVIDUAL
3,EXTRAÇÃO E PREPARAÇÃO DE DADOS,AC Pensamento Critico,20/08/2026,✅,RESPONDIDO,True,INDIVIDUAL
4,INOVAÇÃO E DESIGN THINKING,Roda da inovação,24/08/2026,✅,RESPONDIDO,False,INDIVIDUAL
5,EXTRAÇÃO E PREPARAÇÃO DE DADOS,AC 4 - Coleta Tabela McDonalds,25/08/2026,✅,RESPONDIDO,True,INDIVIDUAL
6,INOVAÇÃO E DESIGN THINKING,Formulário,25/08/2026,✅,RESPONDIDO,False,INDIVIDUAL
7,PROJETO EM CIÊNCIA DE DADOS IV,ATIVIDADE —QUALIDADE DE SOFTWARE,26/08/2026,✅,RESPONDIDO,True,GRUPO
8,EXTRAÇÃO E PREPARAÇÃO DE DADOS,AC 6 - Pontos vs Saldo com Escudo - Serie A Br...,01/09/2026,✅,RESPONDIDO,True,INDIVIDUAL
9,EXTRAÇÃO E PREPARAÇÃO DE DADOS,AC 7 - Nuvem de Palavras Frases Motivacionais,02/09/2026,✅,RESPONDIDO,True,INDIVIDUAL


## 13. Resumo por disciplina

Quantas atividades cada disciplina tem e quantas estão entregues.

In [27]:
resumo = (
    df_atividades.assign(entregues=df_atividades["entregue"].eq("\u2705"))
    .groupby("disciplina")
    .agg(total=("nome", "size"), entregues=("entregues", "sum"))
)
resumo["pendentes"] = resumo["total"] - resumo["entregues"]
resumo

,total,entregues,pendentes
disciplina,,,
EXTRAÇÃO E PREPARAÇÃO DE DADOS,12,12,0
INOVAÇÃO E DESIGN THINKING,2,2,0
PROJETO EM CIÊNCIA DE DADOS IV,1,1,0


## 14. Salvando

Quatro arquivos: disciplinas, grade da semana, atividades e o JSON cru das atividades (já gravado acima).

In [28]:
df.to_csv("turmas-ibmec.csv", index=False, encoding="utf-8")
df_horarios.to_csv("turmas-ibmec-horarios.csv", index=False, encoding="utf-8")
df_atividades.to_csv("atividades-ibmec.csv", index=False, encoding="utf-8-sig")

print(f"turmas-ibmec.csv          -> {len(df)} linhas")
print(f"turmas-ibmec-horarios.csv -> {len(df_horarios)} linhas")
print(f"atividades-ibmec.csv      -> {len(df_atividades)} linhas")

turmas-ibmec.csv          -> 5 linhas
turmas-ibmec-horarios.csv -> 20 linhas
atividades-ibmec.csv      -> 15 linhas


## Conclusão

**Do header** eu tirei tudo que a extração precisava: a URL, a query `status=ATUAL`, o método real
(GET — revelado pelo `access-control-request-method` da captura OPTIONS), o `Authorization: Bearer` e
os headers de navegador. Decodificando o JWT ainda saiu a matrícula e a validade do token.

**Do header, porém, não saía a lista de atividades** — esse endpoint só devolve turmas. Achei as rotas
`/trabalhos/turmas/{idTurma}` e `/trabalhos/{id}?idTurma=...` lendo o próprio JavaScript do portal
(101 chunks do webpack), e a regra do ✅/❌ veio do dicionário de status que o front usa para pintar a
tag na tela — não é interpretação minha.

**Limitação:** o token expira em ~24h. Para reexecutar contra a API é preciso atualizar o `token.txt`
com o `authorization` recém-copiado do DevTools; sem isso o notebook roda com os JSONs em cache.